# Indoor Electrode Analysis — Step-by-Step

This notebook walks through the indoor (or outdoor) electrode importance analysis so you can run each step independently.



- Reuse project code from `indoor_electrode_analysis.py`.

- Keep runs reproducible with fixed seeds.

- Save outputs to the `results/` folder.



Use the sections below to proceed sequentially.

## 1) Initialize Environment and Dependencies

Install dependencies into the selected kernel if needed. If you already set up the project `.venv`, you can skip installation here.



- Tip: Use the VS Code kernel picker to select `Python (eeg-brain-interface)`.

- Tip: If imports fail, run the install cell below.

In [ ]:
# Optional: install requirements into the current kernel

# Remove the leading '!' if you prefer running in a terminal instead of the notebook

# !pip install -r ../requirements.txt



import sys, os, random

import numpy as np

import pandas as pd



# Reproducibility

RANDOM_SEED = 42

random.seed(RANDOM_SEED)

np.random.seed(RANDOM_SEED)



print("Python:", sys.version)

print("Working dir:", os.getcwd())

## 2) Enable Auto-Reload and Set Project Path

The autoreload extension makes it easier to tweak code in `.py` files and re-run without restarting the kernel. We also ensure the repo root is on `sys.path`.

In [ ]:
# Enable autoreload for iterative development

%load_ext autoreload

%autoreload 2



# Ensure project root is in sys.path

import sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if repo_root not in sys.path:

    sys.path.insert(0, repo_root)

print("Repo root added to sys.path:", repo_root)

## 3) Import Libraries and Project Modules

This brings in the scientific Python stack and plotting libs used by the analysis.

In [ ]:
import mne

import matplotlib.pyplot as plt

import seaborn as sns

from pathlib import Path

from collections import defaultdict



from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectKBest, f_classif



sns.set_context("talk")

plt.rcParams["figure.facecolor"] = "white"

print("MNE:", mne.__version__)

## 4) Define Runtime Parameters

Switch between indoor/outdoor and toggle inclusion of 0-back trials.

In [ ]:
ANALYSIS_SESSION = "indoor"  # or "outdoor"

INCLUDE_0_BACK = False

ALL_ELECTRODES = ["Fz", "C4", "Cz", "C3", "Pz", "PO8", "Oz", "PO7"]



RESULTS_DIR = Path("results")

RESULTS_DIR.mkdir(exist_ok=True)



print("Session:", ANALYSIS_SESSION, "| include_0_back:", INCLUDE_0_BACK)

## 5) Discover Participants and Load Data

In [ ]:
from typing import List, Tuple, Dict, Optional



def get_available_participants() -> List[str]:

    processed_dir = Path("results/processed")

    if not processed_dir.exists():

        return []

    parts = [p.name for p in processed_dir.iterdir() if p.is_dir() and not p.name.endswith('.csv')]

    return sorted(parts)



def load_participant_session(participant: str, session_type: str):

    epo_file = Path("results/processed") / participant / f"{session_type}_processed-epo.fif"

    if not epo_file.exists():

        print(f"⚠ Missing file for {participant} {session_type}: {epo_file}")

        return None

    try:

        epochs = mne.read_epochs(epo_file, preload=True, verbose=False)

        if INCLUDE_0_BACK:

            analysis_events = ['0-back', '1-back', '2-back', '3-back']

            event_id_to_difficulty = {epochs.event_id['0-back']: 0,

                                      epochs.event_id['1-back']: 1,

                                      epochs.event_id['2-back']: 2,

                                      epochs.event_id['3-back']: 3}

        else:

            analysis_events = ['1-back', '2-back', '3-back']

            event_id_to_difficulty = {epochs.event_id['1-back']: 1,

                                      epochs.event_id['2-back']: 2,

                                      epochs.event_id['3-back']: 3}

        ep = epochs[analysis_events]

        if len(ep) < 10:

            print(f"⚠ Too few epochs for {participant} {session_type}: {len(ep)}")

            return None

        difficulties = [event_id_to_difficulty[eid] for eid in ep.events[:, 2]]

        meta = pd.DataFrame({

            'difficulty': difficulties,

            'participant': [participant]*len(ep),

            'session_type': [session_type]*len(ep),

        })

        ep.metadata = meta

        print(f"✓ Loaded {participant} {session_type}: {len(ep)} epochs")

        return ep

    except Exception as e:

        print(f"⚠ Error loading {participant} {session_type}: {e}")

        return None



def extract_features(epochs_data, exclude_channels: Optional[List[str]] = None) -> Tuple[np.ndarray, List[str], object]:

    bands = {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 40)}

    ep_filt = epochs_data.copy().filter(4.0, 40.0, picks="eeg")

    if exclude_channels:

        to_drop = [ch for ch in exclude_channels if ch in ep_filt.ch_names]

        if to_drop:

            ep_filt.drop_channels(to_drop)

    try:

        psd = ep_filt.compute_psd(method="welch", fmin=1.0, fmax=40.0,

                                  n_fft=int(ep_filt.info["sfreq"]*2),

                                  n_overlap=int(ep_filt.info["sfreq"]*1),

                                  picks="eeg", verbose=False)

        psds, freqs = psd.get_data(return_freqs=True)

    except Exception:

        from mne.time_frequency import psd_welch

        psds, freqs = psd_welch(ep_filt, fmin=1.0, fmax=40.0,

                                n_fft=int(ep_filt.info["sfreq"]*2),

                                n_overlap=int(ep_filt.info["sfreq"]*1),

                                picks="eeg", average="mean", verbose=False)

    bin_mask = {b: (freqs >= lo) & (freqs < hi) for b,(lo,hi) in bands.items()}

    total_pow = psds.sum(axis=2) + 1e-12

    feat_list, col_names = [], []

    for b, m in bin_mask.items():

        bp = psds[:, :, m].sum(axis=2)

        rel = bp / total_pow

        feat_list.append(rel)

        col_names += [f"{ch}_{b}" for ch in ep_filt.ch_names]

    X = np.concatenate(feat_list, axis=1)

    return X, col_names, ep_filt



def train_and_evaluate_rf(X: np.ndarray, y: np.ndarray, cv_folds: int = 5):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)

    if X_scaled.shape[1] > 20:

        selector = SelectKBest(score_func=f_classif, k=20)

        X_selected = selector.fit_transform(X_scaled, y)

    else:

        X_selected = X_scaled

    min_class = np.bincount(y).min()

    n_splits = min(cv_folds, int(min_class)) if min_class > 1 else 2

    rf = RandomForestClassifier(n_estimators=1000, min_samples_split=4, min_samples_leaf=6,

                               class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1)

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

    scores = cross_val_score(rf, X_selected, y, cv=cv, scoring="accuracy", n_jobs=-1)

    return scores.mean(), scores.std(), scores



def analyze_single_participant_session(participant: str, session_type: str):

    epochs = load_participant_session(participant, session_type)

    if epochs is None:

        return None

    y = epochs.metadata["difficulty"].astype(int).to_numpy()

    results = []

    print("→ Baseline (all electrodes)...")

    X_all, _, _ = extract_features(epochs)

    base_mean, base_std, _ = train_and_evaluate_rf(X_all, y)

    results.append({

        'participant': participant,

        'session_type': session_type,

        'condition': 'baseline',

        'excluded_electrode': 'None',

        'accuracy_mean': base_mean,

        'accuracy_std': base_std,

        'accuracy_drop': 0.0,

    })

    print(f"  Baseline: {base_mean:.3f} ± {base_std:.3f}")

    # Leave-one-out

    print("→ Leave-one-out electrode analysis...")

    for el in ALL_ELECTRODES:

        X_excl, _, _ = extract_features(epochs, exclude_channels=[el])

        if X_excl.size == 0:

            continue

        acc_mean, acc_std, _ = train_and_evaluate_rf(X_excl, y)

        drop = base_mean - acc_mean

        results.append({

            'participant': participant,

            'session_type': session_type,

            'condition': 'leave_one_out',

            'excluded_electrode': el,

            'accuracy_mean': acc_mean,

            'accuracy_std': acc_std,

            'accuracy_drop': drop,

        })

        print(f"  {el}: drop = {drop:.3f}")

    return results

## 6) Execute Analysis for Selected Session

In [ ]:
participants = get_available_participants()

print(f"Found {len(participants)} participants:", participants)



SESSION_TYPES = [ANALYSIS_SESSION]



ALL_RESULTS = []

ELECTRODE_RANKINGS = defaultdict(list)

ELECTRODE_RANK_SUMS = defaultdict(int)



for participant in participants:

    for session_type in SESSION_TYPES:

        res = analyze_single_participant_session(participant, session_type)

        if res:

            ALL_RESULTS.extend(res)



import pandas as pd

if not ALL_RESULTS:

    print("❌ No analyses completed successfully!")

else:

    all_results_df = pd.DataFrame(ALL_RESULTS)

    print(all_results_df.head())

    # Build rankings

    loo = all_results_df[all_results_df['condition']== 'leave_one_out']

    drops_by_el = (loo.groupby('excluded_electrode')['accuracy_drop']

                     .mean()

                     .sort_values(ascending=False))

    rank_order = list(drops_by_el.index)

    for idx, el in enumerate(rank_order, 1):

        ELECTRODE_RANKINGS[el].append(idx)

        ELECTRODE_RANK_SUMS[el] += idx

    print("Rank order:", rank_order)

## 7) Visualize Results

In [ ]:
def create_session_visualization(all_results_df: pd.DataFrame, session_type: str = "indoor"):

    if all_results_df is None or all_results_df.empty:

        print(f"❌ No {session_type} analysis results to visualize!")

        return None

    plt.style.use('default')

    sns.set_palette("Set2")

    fig, ax = plt.subplots(figsize=(12, 8))

    loo = all_results_df[all_results_df['condition']=='leave_one_out'].copy()

    if loo.empty:

        ax.text(0.5, 0.5, f'No leave-one-out data for {session_type}', ha='center', va='center', transform=ax.transAxes)

        return fig

    drops = (loo.groupby('excluded_electrode')['accuracy_drop']

               .mean()

               .sort_values(ascending=False))

    bars = ax.bar(range(len(drops)), drops.values, color=plt.cm.viridis(np.linspace(0,1,len(drops))))

    ax.set_xlabel('Electrode')

    ax.set_ylabel('Mean Accuracy Drop (Higher = More Important)')

    ax.set_title(f'{session_type.title()} — Electrode Importance (LOO)')

    ax.set_xticks(range(len(drops)))

    ax.set_xticklabels(drops.index, rotation=45)

    ax.grid(True, alpha=0.3)

    for i, (el, val) in enumerate(drops.items()):

        ax.text(i, val + 0.001, f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()

    out = RESULTS_DIR / f"{session_type}_electrode_analysis_visualization.png"

    plt.savefig(out, dpi=300, bbox_inches='tight')

    print("Saved:", out)

    plt.show()

    return out



def create_rank_sum_visualization(rank_sum_data: Dict[str,int], session_type: str = "indoor"):

    if not rank_sum_data:

        print(f"❌ No rank sum data to visualize for {session_type}!")

        return None

    items = sorted(rank_sum_data.items(), key=lambda x: x[1])

    fig, ax = plt.subplots(figsize=(12,8))

    labels = [k for k,_ in items]

    vals = [v for _,v in items]

    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(labels)))

    ax.bar(range(len(labels)), vals, color=colors, alpha=0.8)

    ax.set_xlabel('Electrode')

    ax.set_ylabel('Rank Sum (Lower = Better)')

    ax.set_title(f'{session_type.title()} — Rank Sum Summary')

    ax.set_xticks(range(len(labels)))

    ax.set_xticklabels(labels, rotation=45)

    ax.grid(True, alpha=0.3, axis='y')

    for i,(lab,val) in enumerate(zip(labels, vals)):

        ax.text(i, val + max(vals)*0.01, f'{val}', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()

    out = RESULTS_DIR / f"{session_type}_electrode_rank_sum_visualization.png"

    plt.savefig(out, dpi=300, bbox_inches='tight')

    print("Saved:", out)

    plt.show()

    return out



# If we have results, render plots

if 'all_results_df' in globals():

    _ = create_session_visualization(all_results_df, ANALYSIS_SESSION)

    _ = create_rank_sum_visualization(ELECTRODE_RANK_SUMS, ANALYSIS_SESSION)

## 8) Save CSV Outputs

In [ ]:
if 'all_results_df' in globals():

    session_name = ANALYSIS_SESSION

    detailed_path = RESULTS_DIR / f"electrode_analysis_{session_name}_only.csv"

    all_results_df.to_csv(detailed_path, index=False)

    print("Saved:", detailed_path)



    # Ranking summary and rank sums

    if ELECTRODE_RANK_SUMS:

        # Average ranking based on ELECTRODE_RANKINGS

        from statistics import mean, pstdev

        avg_rankings = {}

        for el, ranks in ELECTRODE_RANKINGS.items():

            if ranks:

                avg_rankings[el] = (mean(ranks), pstdev(ranks) if len(ranks) > 1 else 0.0)

        ranking_summary = pd.DataFrame([

            {'electrode': el, 'avg_rank': avg, 'std_rank': std, 'consistency_score': (1/std) if std > 0 else float('inf')}

            for el, (avg, std) in avg_rankings.items()

        ]).sort_values('avg_rank')

        ranking_path = RESULTS_DIR / f"electrode_ranking_{session_name}_only.csv"

        ranking_summary.to_csv(ranking_path, index=False)

        print("Saved:", ranking_path)



        rank_sum_summary = pd.DataFrame([

            {'electrode': el, 'rank_sum': s, 'avg_rank': s / max(1, len(ELECTRODE_RANKINGS.get(el, [])))}

            for el, s in ELECTRODE_RANK_SUMS.items()

        ]).sort_values('avg_rank')

        rank_sum_path = RESULTS_DIR / f"electrode_rank_sums_{session_name}_only.csv"

        rank_sum_summary.to_csv(rank_sum_path, index=False)

        print("Saved:", rank_sum_path)

## 9) Optional: Lightweight Tests and Export

In [ ]:
# Optional: run a quick shape check or a small assertion

if 'all_results_df' in globals():

    assert set(all_results_df['condition'].unique()) >= {'baseline', 'leave_one_out'}

    print('Sanity checks passed.')



# Optional: export to script and capture environment

# !jupyter nbconvert --to script Indoor_Electrode_Analysis.ipynb

# !pip freeze > results/requirements-lock.txt

## 10) Enhanced 8 vs 4 Analysis (Indoor CV → T-Test → Outdoor Testing)

This section is optional. It compares 8 electrodes vs the best 4 (by ranking), tests statistical significance with a paired t-test on CV folds, and evaluates generalization to outdoor data if available.

In [ ]:
from sklearn.base import clone

from sklearn.model_selection import StratifiedKFold

from scipy import stats



def train_and_evaluate_rf_with_models(X, y, cv_folds=5):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)

    selector = None

    if X_scaled.shape[1] > 20:

        selector = SelectKBest(score_func=f_classif, k=20)

        X_selected = selector.fit_transform(X_scaled, y)

    else:

        X_selected = X_scaled

    min_class = np.bincount(y).min()

    n_splits = min(cv_folds, int(min_class)) if min_class > 1 else 2

    rf = RandomForestClassifier(n_estimators=1000, min_samples_split=4, min_samples_leaf=6,

                               class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1)

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

    fold_scores = []

    trained_models = []

    for train_idx, val_idx in cv.split(X_selected, y):

        X_train, X_val = X_selected[train_idx], X_selected[val_idx]

        y_train, y_val = y[train_idx], y[val_idx]

        model = clone(rf)

        model.fit(X_train, y_train)

        fold_scores.append(model.score(X_val, y_val))

        trained_models.append({'model': model, 'scaler': scaler, 'selector': selector})

    return np.mean(fold_scores), np.std(fold_scores), np.array(fold_scores), trained_models



def test_models_on_outdoor_data(trained_models, outdoor_epochs, exclude_channels=None):

    if outdoor_epochs is None:

        return None, None, None

    y_outdoor = outdoor_epochs.metadata["difficulty"].astype(int).to_numpy()

    X_outdoor, _, _ = extract_features(outdoor_epochs, exclude_channels=exclude_channels)

    scores = []

    for info in trained_models:

        Xs = info['scaler'].transform(X_outdoor)

        Xsel = info['selector'].transform(Xs) if info['selector'] is not None else Xs

        scores.append(info['model'].score(Xsel, y_outdoor))

    scores = np.array(scores)

    return scores.mean(), scores.std(), scores



def compare_8vs4_electrodes_with_ttest_and_outdoor_testing(participant, best_4_electrodes):

    indoor_epochs = load_participant_session(participant, "indoor")

    if indoor_epochs is None:

        return None

    y_indoor = indoor_epochs.metadata["difficulty"].astype(int).to_numpy()

    # 8-elec indoor CV

    X8, _, _ = extract_features(indoor_epochs)

    m8, s8, scores8, models8 = train_and_evaluate_rf_with_models(X8, y_indoor)

    # 4-elec indoor CV

    worst4 = [el for el in ALL_ELECTRODES if el not in best_4_electrodes]

    X4, _, _ = extract_features(indoor_epochs, exclude_channels=worst4)

    m4, s4, scores4, models4 = train_and_evaluate_rf_with_models(X4, y_indoor)

    # Paired t-test

    t_stat, p_value = stats.ttest_rel(scores8, scores4)

    # Outdoor test if available

    outdoor_epochs = load_participant_session(participant, "outdoor")

    outdoor_available = outdoor_epochs is not None

    out8_mean = out8_std = out8_scores = None

    out4_mean = out4_std = out4_scores = None

    if outdoor_available:

        out8_mean, out8_std, out8_scores = test_models_on_outdoor_data(models8, outdoor_epochs)

        out4_mean, out4_std, out4_scores = test_models_on_outdoor_data(models4, outdoor_epochs, exclude_channels=worst4)

    return {

        'participant': participant,

        'best_4_electrodes': ', '.join(best_4_electrodes),

        'indoor_cv_8_mean': m8, 'indoor_cv_8_std': s8, 'indoor_cv_8_scores': scores8,

        'indoor_cv_4_mean': m4, 'indoor_cv_4_std': s4, 'indoor_cv_4_scores': scores4,

        't_statistic': t_stat, 'p_value': p_value,

        'outdoor_available': outdoor_available,

        'outdoor_test_8_mean': out8_mean, 'outdoor_test_8_std': out8_std, 'outdoor_test_8_scores': out8_scores,

        'outdoor_test_4_mean': out4_mean, 'outdoor_test_4_std': out4_std, 'outdoor_test_4_scores': out4_scores,

        'indoor_difference_8_minus_4': m8 - m4,

        'generalization_ratio_8': (out8_mean / m8) if outdoor_available and m8 > 0 else None,

        'generalization_ratio_4': (out4_mean / m4) if outdoor_available and m4 > 0 else None,

    }

In [ ]:
# Execute enhanced analysis (optional)

enhanced_results = []

best_4_electrodes = None

if 'all_results_df' in globals() and not all_results_df.empty:

    # Determine best 4 electrodes from current ranking (highest mean drop)

    loo = all_results_df[all_results_df['condition']=='leave_one_out']

    drops = (loo.groupby('excluded_electrode')['accuracy_drop']

               .mean()

               .sort_values(ascending=False))

    best_4_electrodes = list(drops.index[:4])

    print("Best 4 electrodes:", best_4_electrodes)

    # Find participants with both indoor and outdoor files

    participants_with_both = []

    for p in get_available_participants():

        d = Path("results/processed")/p

        if (d/"indoor_processed-epo.fif").exists() and (d/"outdoor_processed-epo.fif").exists():

            participants_with_both.append(p)

    print(f"Participants with both sessions: {participants_with_both}")

    for p in participants_with_both:

        r = compare_8vs4_electrodes_with_ttest_and_outdoor_testing(p, best_4_electrodes)

        if r:

            enhanced_results.append(r)

    print(f"Completed {len(enhanced_results)}/{len(participants_with_both)} enhanced comparisons")

else:

    print("No base results available; run sections 5-8 first.")

## 11) Enhanced Analysis: Summary and Visualization

In [ ]:
def summarize_enhanced_results(enhanced_results):

    if not enhanced_results:

        print("No enhanced results available.")

        return None

    df = pd.DataFrame(enhanced_results)

    print("Indoor CV (mean ± std):")

    print("  8-elec:", df['indoor_cv_8_mean'].mean().round(3), '±', df['indoor_cv_8_std'].mean().round(3))

    print("  4-elec:", df['indoor_cv_4_mean'].mean().round(3), '±', df['indoor_cv_4_std'].mean().round(3))

    print("Significant differences:", int((df['p_value'] < 0.05).sum()), '/', len(df))

    if (df['outdoor_available'] == True).any():

        odf = df[df['outdoor_available'] == True]

        print("Outdoor test (mean ± std):")

        print("  8-elec:", odf['outdoor_test_8_mean'].mean().round(3), '±', odf['outdoor_test_8_std'].mean().round(3))

        print("  4-elec:", odf['outdoor_test_4_mean'].mean().round(3), '±', odf['outdoor_test_4_std'].mean().round(3))

        print("Retention (% of indoor):")

        print("  8-elec:", (odf['generalization_ratio_8'].mean()*100).round(1))

        print("  4-elec:", (odf['generalization_ratio_4'].mean()*100).round(1))

    # Save CSV

    out = RESULTS_DIR / f"enhanced_electrode_comparison_{ANALYSIS_SESSION}.csv"

    pd.DataFrame(enhanced_results).to_csv(out, index=False)

    print("Saved:", out)

    return out



_ = summarize_enhanced_results(enhanced_results)

## 12) Confusion Matrix Analysis (8 vs 4 Electrodes)

Optional diagnostics to compare per-class performance of 8 electrodes vs the best 4 electrodes. This builds cross-validated predictions to compute confusion matrices and summary reports.

In [ ]:
from sklearn.model_selection import cross_val_predict

from sklearn.metrics import confusion_matrix, classification_report



def train_and_evaluate_rf_with_confusion_matrix(X, y, cv_folds=5):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)

    selector = None

    if X_scaled.shape[1] > 20:

        selector = SelectKBest(score_func=f_classif, k=20)

        X_selected = selector.fit_transform(X_scaled, y)

    else:

        X_selected = X_scaled

    min_class = np.bincount(y).min()

    n_splits = min(cv_folds, int(min_class)) if min_class > 1 else 2

    rf = RandomForestClassifier(n_estimators=1000, min_samples_split=4, min_samples_leaf=6,

                               class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1)

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

    y_pred = cross_val_predict(rf, X_selected, y, cv=cv, n_jobs=-1)

    cm = confusion_matrix(y, y_pred, labels=sorted(np.unique(y)))

    scores = cross_val_score(rf, X_selected, y, cv=cv, scoring="accuracy", n_jobs=-1)

    return scores.mean(), scores.std(), scores, cm, y, y_pred



def compare_8vs4_electrodes_with_confusion_matrix(participant, session_type, best_4_electrodes):

    epochs = load_participant_session(participant, session_type)

    if epochs is None:

        return None

    y = epochs.metadata["difficulty"].astype(int).to_numpy()

    diff_map = {0:'0-back', 1:'1-back', 2:'2-back', 3:'3-back'} if INCLUDE_0_BACK else {1:'1-back', 2:'2-back', 3:'3-back'}

    labels = [diff_map[i] for i in sorted(np.unique(y))]

    # 8-elec

    X8, _, _ = extract_features(epochs)

    acc8, std8, _, cm8, y_true, y_pred8 = train_and_evaluate_rf_with_confusion_matrix(X8, y)

    # 4-elec

    worst4 = [el for el in ALL_ELECTRODES if el not in best_4_electrodes]

    X4, _, _ = extract_features(epochs, exclude_channels=worst4)

    acc4, std4, _, cm4, _, y_pred4 = train_and_evaluate_rf_with_confusion_matrix(X4, y)

    return {

        'participant': participant,

        'session_type': session_type,

        'cm_8_electrodes': cm8,

        'cm_4_electrodes': cm4,

        'acc_8': acc8,

        'acc_4': acc4,

        'acc_8_std': std8,

        'acc_4_std': std4,

        'difficulty_labels': labels,

        'y_true': y_true,

        'y_pred_8': y_pred8,

        'y_pred_4': y_pred4,

        'best_4_electrodes': ', '.join(best_4_electrodes),

        'accuracy_difference': acc8 - acc4,

    }

In [ ]:
# Run confusion matrix analysis (optional)

cm_results = []

if 'all_results_df' in globals() and not all_results_df.empty:

    # Use the same best 4 electrodes as in the enhanced section

    if 'best_4_electrodes' not in globals() or not best_4_electrodes:

        loo = all_results_df[all_results_df['condition']=='leave_one_out']

        drops = (loo.groupby('excluded_electrode')['accuracy_drop']

                   .mean()

                   .sort_values(ascending=False))

        best_4_electrodes = list(drops.index[:4])

    print("Best 4 electrodes:", best_4_electrodes)

    participants = get_available_participants()

    for p in participants:

        r = compare_8vs4_electrodes_with_confusion_matrix(p, ANALYSIS_SESSION, best_4_electrodes)

        if r:

            cm_results.append(r)

    print(f"Completed {len(cm_results)}/{len(participants)} confusion matrix analyses")

else:

    print("No base results available; run sections 5-8 first.")

In [ ]:
def create_confusion_matrix_visualization(cm_results, session_type="indoor"):

    if not cm_results:

        print(f"❌ No confusion matrix data to visualize for {session_type}!")

        return None

    n = len(cm_results)

    cols = min(3, n)

    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))

    if n == 1:

        axes = [axes]

    elif rows == 1:

        axes = axes if hasattr(axes, '__len__') else [axes]

    else:

        axes = axes.flatten()

    fig.suptitle(f'Confusion Matrices — {session_type.title()} (8 vs 4)', fontsize=16, fontweight='bold')

    for i, res in enumerate(cm_results):

        ax = axes[i]

        cm8 = res['cm_8_electrodes']

        cm4 = res['cm_4_electrodes']

        diff_labels = res['difficulty_labels']

        combined = np.hstack([cm8, np.zeros((cm8.shape[0], 1)), cm4])

        xticks = [f'{l}\n(8-elec)' for l in diff_labels] + [''] + [f'{l}\n(4-elec)' for l in diff_labels]

        sns.heatmap(combined, annot=True, fmt='d', cmap='Blues', ax=ax,

                    xticklabels=xticks, yticklabels=diff_labels, cbar=True)

        p = res['participant'].replace('sub-', '')

        ax.set_title(f'{p} | acc8={res["acc_8"]:.3f}, acc4={res["acc_4"]:.3f}')

        ax.set_xlabel('Predicted')

        ax.set_ylabel('True')

        ax.axvline(x=len(diff_labels)+0.5, color='red', linewidth=3)

    for j in range(n, len(axes)):

        axes[j].set_visible(False)

    plt.tight_layout()

    out = RESULTS_DIR / f"confusion_matrices_{session_type}.png"

    plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')

    print("Saved:", out)

    plt.show()

    return out



def create_detailed_confusion_matrix_analysis(cm_results, session_type="indoor"):

    if not cm_results:

        print(f"❌ No confusion matrix results for {session_type}!")

        return None

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    fig.suptitle(f'Detailed Confusion Matrix Analysis — {session_type.title()}', fontsize=18, fontweight='bold')

    cm8_list = [r['cm_8_electrodes'] for r in cm_results]

    cm4_list = [r['cm_4_electrodes'] for r in cm_results]

    avg_cm8 = np.mean(cm8_list, axis=0)

    avg_cm4 = np.mean(cm4_list, axis=0)

    labels = cm_results[0]['difficulty_labels']

    sns.heatmap(avg_cm8, annot=True, fmt='.1f', cmap='Blues', ax=axes[0,0], xticklabels=labels, yticklabels=labels)

    axes[0,0].set_title('Average CM — 8 Electrodes')

    sns.heatmap(avg_cm4, annot=True, fmt='.1f', cmap='Reds', ax=axes[0,1], xticklabels=labels, yticklabels=labels)

    axes[0,1].set_title('Average CM — 4 Electrodes')

    # Per-class accuracy

    def per_class_acc(cm):

        return [cm[i,i] / cm[i,:].sum() if cm[i,:].sum() > 0 else 0 for i in range(cm.shape[0])]

    acc8 = np.mean([per_class_acc(cm) for cm in cm8_list], axis=0)

    acc4 = np.mean([per_class_acc(cm) for cm in cm4_list], axis=0)

    x = np.arange(len(labels)); w=0.35

    axes[1,0].bar(x-w/2, acc8, w, label='8-elec', color='steelblue', alpha=0.8)

    axes[1,0].bar(x+w/2, acc4, w, label='4-elec', color='lightcoral', alpha=0.8)

    axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(labels)

    axes[1,0].set_title('Per-Class Accuracy (avg)'); axes[1,0].legend(); axes[1,0].grid(axis='y', alpha=0.3)

    # Difference heatmap

    diff = avg_cm8 - avg_cm4

    sns.heatmap(diff, annot=True, fmt='.1f', cmap='RdBu_r', center=0, ax=axes[1,1], xticklabels=labels, yticklabels=labels)

    axes[1,1].set_title('CM Difference (8 - 4)')

    plt.tight_layout()

    out = RESULTS_DIR / f"detailed_confusion_matrix_analysis_{session_type}.png"

    plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')

    print("Saved:", out)

    plt.show()

    return out



# Visualize and export summaries

_ = create_confusion_matrix_visualization(cm_results, ANALYSIS_SESSION)

_ = create_detailed_confusion_matrix_analysis(cm_results, ANALYSIS_SESSION)



# Export compact summary CSV

if cm_results:

    cm_summary = []

    for r in cm_results:

        cm_summary.append({

            'participant': r['participant'],

            'session_type': r['session_type'],

            'accuracy_8_electrodes': r['acc_8'],

            'accuracy_4_electrodes': r['acc_4'],

            'accuracy_difference': r['accuracy_difference'],

            'best_4_electrodes': r['best_4_electrodes'],

        })

    cm_df = pd.DataFrame(cm_summary)

    path = RESULTS_DIR / f"confusion_matrix_summary_{ANALYSIS_SESSION}.csv"

    cm_df.to_csv(path, index=False)

    print("Saved:", path)